# Gemma 4 Vision Testing

**WeatherSpeak PH** — Gemma 4 Hackathon

## About Gemma 4 Vision

Gemma 4 is Google's open-weight multimodal language model. For this experiment:
- **Model**: Gemma 4 26B (multimodal)
- **Runtime**: Ollama (local inference)
- **Approach**: Vision-first OCR (no specialized OCR engine)

### The Hypothesis

Can a general-purpose vision-language model **replace specialized OCR** for structured government documents like PAGASA bulletins?

**Potential advantages:**
- ✅ Semantic understanding (not just character recognition)
- ✅ Natural table and structure handling
- ✅ Can extract specific fields via prompts
- ✅ Same model for OCR + translation (simpler pipeline)
- ✅ Aligns with hackathon theme (Gemma 4 throughout)

**This is the critical experiment** — if Gemma 4 Vision works well, it transforms our entire approach.

## 1. Prerequisites

**Before running this notebook**, ensure:
1. Ollama is installed: `brew install ollama` (macOS) or download from [ollama.ai](https://ollama.ai)
2. Gemma 4 model is pulled: `ollama pull gemma2:27b-vision` (or 26B variant)
3. Ollama server is running: `ollama serve`

In [ ]:
import os
import json
import base64
from pathlib import Path
from PIL import Image
import time
import requests

print("✓ Imports successful")

## 2. Test Ollama Connection

Verify Ollama is running and Gemma 4 model is available.

In [ ]:
OLLAMA_API = "http://localhost:11434"
MODEL_NAME = "gemma2:27b-vision"  # Adjust if using different variant

# Test connection
try:
    response = requests.get(f"{OLLAMA_API}/api/tags")
    models = response.json()['models']
    model_names = [m['name'] for m in models]
    
    print(f"✓ Ollama is running")
    print(f"\nAvailable models:")
    for name in model_names:
        print(f"  - {name}")
    
    if any(MODEL_NAME in name for name in model_names):
        print(f"\n✓ {MODEL_NAME} is available")
    else:
        print(f"\n⚠️  {MODEL_NAME} not found. Run: ollama pull {MODEL_NAME}")
except Exception as e:
    print(f"✗ Error connecting to Ollama: {e}")
    print(f"\nMake sure Ollama is running: ollama serve")

## 3. Load Sample Data

Load the same sample images used for Surya and PaddleOCR.

In [ ]:
# Load metadata
data_dir = Path("../data")
metadata_path = data_dir / "sample_metadata.json"

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

samples = metadata['samples']
print(f"Loaded {len(samples)} sample images")

# Create output directory for results
output_dir = data_dir / "gemma4_results"
output_dir.mkdir(exist_ok=True)

## 4. Helper Function: Image to Base64

Ollama API requires images to be base64-encoded.

In [ ]:
def encode_image_to_base64(image_path: str) -> str:
    """Convert image file to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

print("✓ Helper function defined")

## 5. Create Prompts for OCR Extraction

We'll use two prompting strategies:
1. **Simple extraction**: "Extract all text from this image"
2. **Structured extraction**: "Extract storm name, category, coordinates, warnings..."

In [ ]:
# Prompt for general text extraction
SIMPLE_PROMPT = """Extract all visible text from this PAGASA typhoon bulletin image.
Preserve the original formatting, line breaks, and structure as much as possible.
Include headers, body text, tables, and any warnings or affected areas."""

# Prompt for structured extraction
STRUCTURED_PROMPT = """This is a PAGASA typhoon bulletin. Extract the following information:

1. Storm name and category
2. Date and time of issuance
3. Storm location (latitude/longitude)
4. Maximum sustained winds
5. Movement (direction and speed)
6. Affected areas/provinces
7. Any warnings or advisories
8. Full text content (preserve all text from the document)

Format as JSON with clear sections."""

print("✓ Prompts defined")

## 6. Run Gemma 4 Vision on Sample Images

Process each sample using the simple extraction prompt.

In [ ]:
def call_gemma4_vision(image_path: str, prompt: str) -> dict:
    """Call Gemma 4 Vision via Ollama API."""
    image_b64 = encode_image_to_base64(image_path)
    
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "images": [image_b64],
        "stream": False
    }
    
    response = requests.post(
        f"{OLLAMA_API}/api/generate",
        json=payload
    )
    
    return response.json()

print("✓ API function defined")

In [ ]:
results = []

for i, sample in enumerate(samples, 1):
    print(f"\n{'='*60}")
    print(f"Processing {i}/{len(samples)}: {sample['filename']}")
    print(f"{'='*60}")
    
    try:
        # Run vision model
        start_time = time.time()
        response = call_gemma4_vision(sample['image_path'], SIMPLE_PROMPT)
        processing_time = time.time() - start_time
        
        # Extract response
        extracted_text = response.get('response', '')
        
        # Store results
        result = {
            'filename': sample['filename'],
            'storm': sample['storm'],
            'processing_time': processing_time,
            'full_text': extracted_text,
            'success': True
        }
        results.append(result)
        
        # Display summary
        print(f"\n📊 Results:")
        print(f"  Processing time: {processing_time:.2f}s")
        print(f"  Characters extracted: {len(extracted_text)}")
        print(f"\n📄 Extracted text (first 500 characters):")
        print("-" * 60)
        print(extracted_text[:500])
        print("-" * 60)
        
    except Exception as e:
        print(f"✗ Error processing {sample['filename']}: {e}")
        results.append({
            'filename': sample['filename'],
            'storm': sample['storm'],
            'processing_time': 0,
            'full_text': '',
            'error': str(e),
            'success': False
        })

## 7. Save Results

Save Gemma 4 Vision results for comparison.

In [ ]:
# Save results to JSON
results_path = output_dir / "gemma4_vision_results.json"
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to: {results_path}")

# Also save full text files for easy reading
for result in results:
    if result.get('success', False):
        text_path = output_dir / f"{Path(result['filename']).stem}_text.txt"
        with open(text_path, 'w', encoding='utf-8') as f:
            f.write(result['full_text'])

print(f"✓ Text files saved to: {output_dir}")

## 8. Performance Summary

Aggregate performance metrics.

In [ ]:
import statistics

successful_results = [r for r in results if r.get('success', False)]
processing_times = [r['processing_time'] for r in successful_results]

print("\n" + "="*60)
print("GEMMA 4 VISION PERFORMANCE SUMMARY")
print("="*60)
print(f"\n📊 Processing:")
print(f"  Successful: {len(successful_results)}/{len(results)}")
if processing_times:
    print(f"  Average time: {statistics.mean(processing_times):.2f}s")
    print(f"  Min: {min(processing_times):.2f}s")
    print(f"  Max: {max(processing_times):.2f}s")
print(f"\n✓ All results saved to: {output_dir.absolute()}")

## 9. Visual Inspection

Display one sample with its extracted text for manual quality assessment.

In [ ]:
# Display first successful sample
successful = [r for r in results if r.get('success', False)]
if successful:
    sample_result = successful[0]
    sample_info = next(s for s in samples if s['filename'] == sample_result['filename'])
    
    print(f"Sample: {sample_result['filename']}")
    print(f"Storm: {sample_result['storm']}")
    print(f"\n" + "="*60)
    
    # Show image
    img = Image.open(sample_info['image_path'])
    display(img.resize((600, int(600 * img.size[1] / img.size[0]))))
    
    # Show extracted text
    print("\n📄 EXTRACTED TEXT:")
    print("="*60)
    print(sample_result['full_text'])
    print("="*60)

## 10. Test Structured Extraction (Optional)

Try the structured prompt on one sample to see if Gemma 4 can extract specific fields.

In [ ]:
# Run structured extraction on first sample
if samples:
    print("Testing structured extraction on first sample...\n")
    
    sample = samples[0]
    response = call_gemma4_vision(sample['image_path'], STRUCTURED_PROMPT)
    structured_text = response.get('response', '')
    
    print("📋 STRUCTURED EXTRACTION:")
    print("="*60)
    print(structured_text)
    print("="*60)

## 11. Preliminary Assessment

### ✅ Potential Strengths
- Semantic understanding of document structure
- Can extract specific fields via prompts
- Same model for OCR + translation
- Aligns with hackathon theme

### ⚠️ Considerations
- Processing speed vs specialized OCR?
- Character-level accuracy?
- Consistency across different bulletin formats?
- Hallucination risk?

### 📝 Critical Questions
1. **Accuracy**: Is text extraction accurate enough for translation?
2. **Structure**: Does it preserve tables, coordinates, warnings correctly?
3. **Reliability**: Does it work consistently across all samples?
4. **Speed**: Is processing time acceptable for production?

### 🎯 Next Steps
Proceed to **Notebook 05** for comprehensive comparison across all three approaches:
- Surya OCR
- PaddleOCR
- Gemma 4 Vision

**The decision will determine our entire production pipeline.**